In [ ]:
import requests
import json
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup


def clean_text(html):
    """Remove HTML tags and extra spaces from text."""
    return BeautifulSoup(html, "html.parser").get_text(separator=" ").strip()


def extract_info_from_paragraph(paragraph_html):
    """Extract address, phone, hours, and other_info from a paragraph."""
    phone = "Not available"
    hours = "Not available"
    address = "Not available"
    other_info_parts = []

    # Kansas area codes for phone number detection
    kansas_area_codes = ["316", "620", "785", "913"]
    phone_pattern = re.compile(rf"\b({'|'.join(kansas_area_codes)})[-.\s]?\d{{3}}[-.\s]?\d{{4}}\b")

    # ZIP code and KS pattern
    zip_code_pattern = re.compile(r"\b\d{5}\b") 
    ks_pattern = re.compile(r" KS\b") 

    # Clean text and split 
    soup = BeautifulSoup(paragraph_html, "html.parser")
    paragraph_text = clean_text(paragraph_html)

    # Extract address
    address_candidates = []
    for part in paragraph_text.split("\n"):
        if ks_pattern.search(part) or zip_code_pattern.search(part):
            address_candidates.append(part.strip())
    address = address_candidates[0] if address_candidates else "Not available"

    # Extract phone number
    phone_matches = phone_pattern.findall(paragraph_text)
    if phone_matches:
        phone = phone_matches[0]  # Use the first match

    # Extract hours
    hours_candidates = []
    for part in paragraph_text.split("\n"):
        if re.search(r"\b(\d{1,2}(:\d{2})?\s?(AM|PM|am|pm)|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b", part, re.IGNORECASE):
            if phone not in part and address not in part:
                hours_candidates.append(part.strip())
    hours = hours_candidates[0] if hours_candidates else "Not available"

    # Extract links
    links = [a["href"] for a in soup.find_all("a", href=True)]

    # Extract OTHER INFO 
    other_info_parts = []
    for part in paragraph_text.split("\n"):
        part = part.strip()
        if part and part not in {address, phone, hours}:
            other_info_parts.append(part)

    # Include links in other_info
    other_info_parts.extend(links)

    # Remove duplicates from other_info
    other_info = ", ".join(list(set(other_info_parts))).strip()

    return phone, hours, other_info, address


# Fetch county data from the U.S. Census API
url = "https://api.census.gov/data/2020/dec/pl?get=NAME&for=county:*&in=state:20"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    options = Options()
    options.add_argument("--headless")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

    driver = webdriver.Edge(options=options)
    county_pantries = {}

    for entry in data[1:]:  # Skip the header row
        county_name = entry[0]
        cleaned_county_name = county_name.replace(",", "").replace(" ", "-").lower().replace("-kansas", "")
        county_url = f"https://kansasfoodsource.org/category/help-agency/{cleaned_county_name}/"
        print(f"\nVisiting: {county_url}")

        try:
            driver.get(county_url)
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, "h2.entry-title.ast-blog-single-element a")))
            pantry_elements = driver.find_elements(By.CSS_SELECTOR, "h2.entry-title.ast-blog-single-element a")
            pantry_names = [pantry.text.strip() for pantry in pantry_elements]
            pantry_links = [pantry.get_attribute("href") for pantry in pantry_elements]

            county_pantries[county_name] = []

            # Track processed paragraphs to avoid duplicates
            processed_paragraphs = set()

            for pantry_name, pantry_link in zip(pantry_names, pantry_links):
                print(f"  → Fetching details for {pantry_name}")

                try:
                    driver.get(pantry_link)
                    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.entry-content.clear")))
                    content_element = driver.find_element(By.CSS_SELECTOR, "div.entry-content.clear")
                    paragraphs = content_element.find_elements(By.TAG_NAME, "p")

                    if paragraphs:
                        first_paragraph = paragraphs[0].get_attribute("innerHTML")

                        # Skip if this paragraph has already been processed
                        if first_paragraph in processed_paragraphs:
                            continue

                        processed_paragraphs.add(first_paragraph)  # Mark as processed

                        # Extract phone, hours, other_info, and address
                        phone, hours, other_info, address = extract_info_from_paragraph(first_paragraph)

                        county_pantries[county_name].append({
                            "pantry_name": pantry_name,
                            "address": address,
                            "phone": phone,
                            "hours": hours,
                            "other_info": other_info,
                            "link": pantry_link
                        })
                except Exception as e:
                    print(f"    Skipping {pantry_name}: {e}")

            # Save JSON after each county
            with open("10.json", "w", encoding="utf-8") as f:
                json.dump(county_pantries, f, indent=4)

        except Exception as e:
            print(f"Skipping {county_name}: {e}")
            county_pantries[county_name] = []

    driver.quit()
    print("\nData extraction completed. Results saved to 10.json.")

else:
    print(f"Error fetching data from API. Status code: {response.status_code}")